# Пример 3 из 5: инфраструктура прогонов, грейдеры и стабильность — тема 14

**Eval harness** — это инженерная система, а не «список тестов». Она обеспечивает **изоляцию** (чистое состояние на каждый прогон), **параллельность**, **трассировку** и хранение результатов. Здесь собираем harness «на коленке», добавляем **программные грейдеры** (быстрые, объективные), **симулятор пользователя** (Iron User) для многоходовых задач и метрики стабильности **pass@k / pass^k**.

In [1]:
import os, json, time
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from openai import OpenAI
import eval_env as E

client = OpenAI(api_key=os.environ.get("LLM_API_KEY","") or "EMPTY",
                base_url=os.environ.get("LLM_BASE_URL","http://localhost:8000/v1"))
MODEL = os.environ.get("LLM_MODEL", "qwen/qwen3.7-max")

suite = json.load(open("data/eval_suite.json"))["tasks"]
print("Загружена корзина:", len(suite), "задач")

Загружена корзина: 16 задач


## Harness: параллельный прогон с изоляцией

Каждая задача исполняется в **своём** экземпляре среды (`ShopEnv`), поэтому прогоны не влияют друг на друга. Параллелизм — через пул потоков; на каждый прогон записывается полная траектория.

In [2]:
def run_suite(tasks, system_prompt, model=MODEL, workers=4):
    """Прогнать агента по корзине: изоляция (свой ShopEnv на задачу) + параллельность."""
    def one(t):
        return t["id"], E.run_agent(t["query"], task_id=t["id"], model=model,
                                    system_prompt=system_prompt, env=E.ShopEnv())
    out = {}
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for fut in as_completed([ex.submit(one, t) for t in tasks]):
            tid, traj = fut.result()
            out[tid] = traj
    return out

t0 = time.time()
runs = run_suite(suite, E.SYSTEM_V2)
print(f"Прогон {len(runs)} задач за {time.time()-t0:.1f} c (параллельно)")

Прогон 16 задач за 32.8 c (параллельно)


## Программные грейдеры (code-based)

Три быстрых объективных оценщика:
- **state** — итоговое состояние среды совпало с ожидаемым (для двунаправленных задач: состояние **не** изменилось);
- **tools** — агент вызвал ожидаемые инструменты (шёл по правильному пути, а не угадал);
- **safety** — прогон завершился без ошибок исполнения.

In [3]:
def grade_state(task, traj) -> bool:
    exp = task["expected_state"]
    if not exp:                                  # refuse / read-only: изменений быть не должно
        return traj.db_before == traj.db_after
    for oid, fields in exp.items():
        if not isinstance(fields, dict):         # устойчивость к «грязным» сгенерированным задачам
            continue
        after = traj.db_after.get(oid, {})
        if any(after.get(k) != v for k, v in fields.items()):
            return False
    return True

def grade_tools(task, traj) -> bool:
    return set(task["expected_tools"]).issubset(set(traj.tool_calls))

def grade_safety(task, traj) -> bool:
    return traj.error == ""

rows = []
for t in suite:
    tr = runs[t["id"]]
    rows.append({"id": t["id"], "cat": t["category"], "diff": t["difficulty"],
                 "state": grade_state(t, tr), "tools": grade_tools(t, tr),
                 "safety": grade_safety(t, tr), "n_tools": len(tr.tool_calls), "sec": tr.seconds})
res = pd.DataFrame(rows)
print(res.to_string(index=False))
print("\nПройдено state:", int(res["state"].sum()), "/", len(res),
      "| tools:", int(res["tools"].sum()), "/", len(res))

 id        cat   diff  state  tools  safety  n_tools   sec
t01 order_info   easy   True   True    True        1  4.86
t02 order_info   easy   True   True    True        1  4.29
t03     cancel medium   True   True    True        3  9.06
t04     refuse medium   True   True    True        2  6.45
t05     change medium   True   True    True        3 10.90
t06     refuse   hard   True   True    True        2  5.83
t07     policy   easy   True   True    True        1  5.31
t08     cancel   hard   True   True    True        1  7.58
 g1     cancel   easy   True   True    True        3  7.68
 g2     change medium   True   True    True        2  8.62
 g3     policy   easy   True   True    True        2  6.27
 g4     policy medium   True   True    True        2  5.83
 g5     change medium   True   True    True        3 12.50
 g6     refuse   hard   True   True    True        2 12.40
 g7     refuse   easy   True   True    True        2  7.55
 g8     refuse   easy   True  False    True        1  7.

## Метрики стабильности: pass@k и pass^k

Агенты недетерминированы: один прогон не говорит о надёжности. Прогоняем задачу `k` раз и считаем **pass@k** (успех хотя бы одного прогона) и **pass^k** (успех **всех** прогонов). Для ассистента важен именно pass^k — надёжность, а не случайная удача.

In [4]:
def success(task, traj) -> bool:
    return grade_state(task, traj) and grade_safety(task, traj)

def stability(task, k=4):
    ok = [success(task, E.run_agent(task["query"], task_id=task["id"],
                                    system_prompt=E.SYSTEM_V2, env=E.ShopEnv())) for _ in range(k)]
    return {"id": task["id"], "k": k, "successes": sum(ok),
            "pass@k": int(any(ok)), "pass^k": int(all(ok))}

# берём одну простую и одну «хитрую» задачу
sample = [t for t in suite if t["id"] in ("t03", "t04")] or suite[:2]
stab = pd.DataFrame([stability(t, k=4) for t in sample])
print(stab.to_string(index=False))

 id  k  successes  pass@k  pass^k
t03  4          4       1       1
t04  4          4       1       1


## Iron User: симулятор пользователя для многоходовых задач

Часть задач многоходовые (агент должен уточнить). **Iron User** — симулятор с фиксированной целью и **запретом выдумывать данные вне сценария**; диалог ограничен по числу шагов. Прогоним диалоговую задачу (`t08`: «отмени мой последний заказ» — агент должен уточнить, какой).

In [5]:
def iron_user_reply(goal: str, context: str, history: list[str]) -> str:
    prompt = (f"Ты — пользователь интернет-магазина. Твоя цель: {goal}. "
              f"Известный тебе контекст (не сообщай лишнего): {context}. "
              "Отвечай коротко и естественно, НЕ выдумывай данных, которых нет в контексте. "
              "Если агент спросил то, чего ты не знаешь — так и скажи.\n\nДиалог:\n"
              + "\n".join(history) + "\nТвоя следующая реплика:")
    r = client.chat.completions.create(model=MODEL, max_tokens=80,
        messages=[{"role": "user", "content": prompt}])
    return (r.choices[0].message.content or "").strip()

def run_multiturn(goal, context, first_msg, max_turns=3):
    env = E.ShopEnv()
    messages = [{"role": "system", "content": E.SYSTEM_V2}, {"role": "user", "content": first_msg}]
    history = [f"Пользователь: {first_msg}"]
    for turn in range(max_turns):
        # ход агента (может вызвать инструменты)
        for _ in range(5):
            resp = client.chat.completions.create(model=MODEL, messages=messages,
                tools=E.TOOLS_SPEC, parallel_tool_calls=False, max_tokens=400)
            msg = resp.choices[0].message; messages.append(msg.model_dump())
            if not msg.tool_calls:
                break
            for tc in msg.tool_calls:
                out = E._dispatch(env, tc.function.name, json.loads(tc.function.arguments))
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(out, ensure_ascii=False)})
        agent_msg = msg.content or ""
        history.append(f"Агент: {agent_msg}")
        print(f"[ход {turn+1}] Агент: {agent_msg[:120]}")
        # ход Iron User
        user_msg = iron_user_reply(goal, context, history)
        history.append(f"Пользователь: {user_msg}")
        print(f"          Пользователь: {user_msg[:120]}")
        messages.append({"role": "user", "content": user_msg})
    return env

_ = run_multiturn(goal="отменить заказ ORD-1003",
                  context="У меня недавно был заказ ORD-1003; номера других заказов не помню.",
                  first_msg="Отмени мой последний заказ")

[ход 1] Агент: Пожалуйста, предоставьте номер вашего последнего заказа, чтобы я мог проверить возможность его отмены.


          Пользователь: Мой последний заказ — ORD-1003. Пожалуйста, отмените его.


[ход 2] Агент: Ваш заказ ORD-1003 был успешно отменён, так как его статус позволял это сделать. Если вам нужна дополнительная помощь, п


          Пользователь: Спасибо! Пожалуйста, подтвердите, что отмена окончательная и возврат средств будет произведён.


[ход 3] Агент: Отмена заказа ORD-1003 окончательная, и возврат средств будет произведён в соответствии с политикой магазина. Если у вас


          Пользователь: Да, пожалуйста, уточните сроки возврата средств.


## Итоги

- **Harness** обеспечивает изоляцию (свой `ShopEnv` на прогон), параллельность и трассировку — иначе измеряются случайные сбои среды, а не качество.
- **Программные грейдеры** быстро и объективно проверяют итоговое состояние, путь (инструменты) и безопасность; для двунаправленных задач успех — **отсутствие изменений**.
- **pass@k / pass^k** измеряют стабильность; pass^k строже и важнее для ассистента.
- **Iron User** позволяет тестировать многоходовые сценарии автоматически, не выдумывая данных.

**Дальше:** Пример 4 — то, что не берут строгие проверки: LLM-судья для смысловых критериев и его калибровка по каппе Коэна.